# 🏆 District Accord

Trains an LLM agent through **generational league self-play**:
each generation of the model plays against a pool of all past checkpoints,
creating progressively harder opponents and driving recursive skill improvement.

## Architecture
```
League Pool: [Gen0, Gen1, Gen2, ...]
                    ↓
Training agent (District 0) ← GRPO updates
Frozen opponents (Districts 1-11) ← sampled from league pool
```

## Requirements
- Google Colab T4 GPU (16GB VRAM)
- Previous GRPO checkpoints in `outputs/grpo_checkpoints/`
- Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ============================================================
# CELL 1: Install & Clone
# ============================================================
# Install all dependencies. The project is installed in editable
# mode so district_accord can be imported directly.

!pip install unsloth trl datasets matplotlib seaborn --quiet
!git clone https://github.com/tezivindh/meta-env-hackathon-final.git
%cd meta-env-hackathon-final
!pip install -e . --quiet
print('✅ Setup complete')

In [ ]:
# ============================================================
# CELL 2: Imports & Global Config
# ============================================================

import os
import json
import time
import random
import traceback
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

# district_accord imports
from district_accord.env import DistrictAccordEnv
from district_accord.utils.config import EnvConfig
from district_accord.spaces.action_parser import ActionParser
from district_accord.policy.self_play import SelfPlayPolicy

# ── League self-play config ───────────────────────────────────────────────
MODEL_NAME          = "unsloth/Qwen2.5-1.5B-Instruct"
NUM_GENERATIONS     = 4          # number of self-play generations to train
EPISODES_PER_GEN    = 25         # episodes per generation for GRPO data
EVAL_EPISODES       = 10         # evaluation episodes after each generation
EVAL_SEEDS          = list(range(200, 210))  # fixed seeds for fair comparison
TRAIN_AGENT_ID      = 0          # district controlled by the LLM being trained
CHECKPOINT_BASE     = "outputs/grpo_checkpoints"  # pre-trained GRPO checkpoints
SELFPLAY_BASE       = "outputs/selfplay_checkpoints"
LEAGUE_STATE_PATH   = "outputs/league_state.json"

# Environment config
ENV_CFG = EnvConfig(num_districts=12, max_turns=100)

# Valid action strings
VALID_ACTIONS = [
    "invest", "defend", "ignore", "recover",
    "request_aid", "share", "propose", "accept", "reject",
]

os.makedirs(SELFPLAY_BASE, exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print('✅ Config loaded')
print(f'   Generations: {NUM_GENERATIONS}')
print(f'   Episodes/gen: {EPISODES_PER_GEN}')
print(f'   Eval episodes: {EVAL_EPISODES}')
print(f'   Train agent: District {TRAIN_AGENT_ID}')

In [ ]:
# ============================================================
# CELL 3: VRAM Helper
# ============================================================
# VRAM on T4 is precious. We track usage before/after each phase.
# Only ONE frozen opponent model is loaded at a time alongside
# the training model.

def vram_gb() -> float:
    """Return current GPU VRAM usage in GB."""
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / 1e9


def vram_report(label: str = "") -> None:
    """Print current VRAM usage."""
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'  [VRAM{" - " + label if label else ""}] {used:.2f} / {total:.2f} GB')


def free_frozen_model(model, label: str = "") -> None:
    """
    Safely delete a frozen opponent model and free VRAM.
    Should be called after frozen inference is complete.
    """
    del model
    torch.cuda.empty_cache()
    if label:
        print(f'  🗑  Freed frozen model [{label}]', end=' ')
    vram_report()


vram_report('startup')
print('✅ VRAM helper ready')

In [ ]:
# ============================================================
# CELL 4: Load Training Model (Unsloth)
# ============================================================
# The TRAINING model is loaded once and kept in memory throughout.
# We use Unsloth for 4-bit QLoRA efficiency.
# Frozen OPPONENT models load/unload separately (Cell 6).

from unsloth import FastLanguageModel

vram_report('before model load')

train_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)

train_model = FastLanguageModel.get_peft_model(
    train_model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vram_report('after model load')
print('\n✅ Training model loaded (Qwen2.5-1.5B-Instruct, 4-bit QLoRA)')

In [ ]:
# ============================================================
# CELL 5: GenerationLeague Class
# ============================================================
# Maintains a pool of all past model checkpoints.
# Implements weighted sampling that favors recent generations
# (recent gens are 2x as likely as older ones).

@dataclass
class GenerationEntry:
    """Metadata for one generation in the league."""
    gen_num: int           # generation index (0 = base)
    checkpoint_path: str   # path to checkpoint directory
    avg_eval_reward: float = 0.0  # filled in after evaluation
    collapse_rate: float  = 0.0  # fraction of agents that collapsed


class GenerationLeague:
    """
    Pool of past model checkpoints for league self-play.

    Sampling strategy: 'weighted_recent' gives recent generations
    2x the sampling weight of older ones, creating pressure to
    improve against the most capable recent opponents.
    """

    def __init__(self, state_path: str = LEAGUE_STATE_PATH):
        self.generations: List[GenerationEntry] = []
        self.state_path = state_path
        # Track which opponent gens were sampled each generation
        self.opponent_samples: Dict[int, List[int]] = {}  # gen -> [opponent_gen_ids]

    def add_generation(self, gen_num: int, checkpoint_path: str,
                        avg_eval_reward: float = 0.0,
                        collapse_rate: float = 0.0) -> None:
        """
        Register a new checkpoint in the league pool.

        Args:
            gen_num: generation index
            checkpoint_path: path to saved model checkpoint
            avg_eval_reward: evaluation reward for this generation
            collapse_rate: fraction of episodes where agent collapsed
        """
        entry = GenerationEntry(
            gen_num=gen_num,
            checkpoint_path=checkpoint_path,
            avg_eval_reward=avg_eval_reward,
            collapse_rate=collapse_rate,
        )
        self.generations.append(entry)
        self.save()
        print(f'  [League] Added Gen {gen_num}: {checkpoint_path}')

    def sample_opponent(self, strategy: str = "weighted_recent") -> GenerationEntry:
        """
        Sample one generation from the pool to use as opponent.

        Strategies:
          'uniform'         — equal probability for all generations
          'weighted_recent' — recent generations are 2x more likely

        Returns:
            GenerationEntry with checkpoint_path to load.
        """
        if not self.generations:
            raise ValueError("League pool is empty — add at least one generation first.")

        n = len(self.generations)

        if strategy == "uniform" or n == 1:
            return random.choice(self.generations)

        # weighted_recent: recent half gets 2x weight
        split = max(1, n // 2)
        old_gens  = self.generations[:n - split]   # older half
        new_gens  = self.generations[n - split:]   # newer half

        # Build weights: 1.0 for old, 2.0 for new
        weights = [1.0] * len(old_gens) + [2.0] * len(new_gens)
        total   = sum(weights)
        probs   = [w / total for w in weights]

        chosen = np.random.choice(len(self.generations), p=probs)
        return self.generations[chosen]

    def record_sample(self, current_gen: int, opponent_gen: int) -> None:
        """Record which opponent generation was sampled (for diversity plot)."""
        if current_gen not in self.opponent_samples:
            self.opponent_samples[current_gen] = []
        self.opponent_samples[current_gen].append(opponent_gen)

    def get_all_generations(self) -> List[GenerationEntry]:
        """Return list of all registered generations."""
        return list(self.generations)

    def save(self) -> None:
        """Persist league state to JSON (called after every update)."""
        state = {
            "generations": [
                {
                    "gen_num": g.gen_num,
                    "checkpoint_path": g.checkpoint_path,
                    "avg_eval_reward": g.avg_eval_reward,
                    "collapse_rate": g.collapse_rate,
                }
                for g in self.generations
            ],
            "opponent_samples": {str(k): v for k, v in self.opponent_samples.items()},
        }
        os.makedirs(os.path.dirname(self.state_path), exist_ok=True)
        with open(self.state_path, "w") as f:
            json.dump(state, f, indent=2)

    @classmethod
    def load(cls, state_path: str = LEAGUE_STATE_PATH) -> "GenerationLeague":
        """Restore league from saved JSON state."""
        league = cls(state_path=state_path)
        if not os.path.exists(state_path):
            return league
        with open(state_path) as f:
            state = json.load(f)
        for g in state.get("generations", []):
            league.generations.append(GenerationEntry(**g))
        league.opponent_samples = {
            int(k): v for k, v in state.get("opponent_samples", {}).items()
        }
        return league

    def __repr__(self) -> str:
        return f"GenerationLeague({len(self.generations)} gens)"


# Initialise the league and register Gen 0 from earliest GRPO checkpoint
league = GenerationLeague()

gen0_path = os.path.join(CHECKPOINT_BASE, "checkpoint-100")
if os.path.exists(gen0_path):
    league.add_generation(0, gen0_path)
    print(f'✅ League initialised with Gen 0: {gen0_path}')
else:
    # Fall back: use base model name as Gen 0
    league.add_generation(0, MODEL_NAME)
    print(f'⚠  Gen 0 checkpoint not found, using base model as Gen 0')

print(f'   League pool: {league}')

In [ ]:
# ============================================================
# CELL 6: Frozen Opponent Inference
# ============================================================
# Loads a checkpoint as a frozen model (no gradients),
# runs ONE inference call, then the caller is responsible
# for calling free_frozen_model() to release VRAM.
#
# Design: only ONE frozen model in memory at a time.
# All frozen opponents share the same tokenizer as the train model.

SYSTEM_PROMPT = """You are a district in a multi-agent crisis management environment with 12 districts.
Each turn, choose exactly ONE action to maximize survival and cooperation over 100 turns.
Reply with ONLY the action name. Nothing else."""


def obs_to_prompt(obs_dict: dict, agent_id: int, env: DistrictAccordEnv) -> str:
    """
    Convert an agent's observation dict into a natural language prompt.
    Works with the actual district_accord dict observation format.
    """
    s    = obs_dict["self"]        # [resources, stability, crisis_exposure, stability_delta]
    c    = obs_dict["crisis"]      # [crisis_level]
    t    = obs_dict["turn"]        # [turn_progress 0..1]
    mask = obs_dict["action_mask"] # [9] binary
    others = obs_dict["others"]    # list of peer state vectors

    avg_peer_res  = float(np.mean([o[0] for o in others])) if others else 0.0
    avg_peer_stab = float(np.mean([o[1] for o in others])) if others else 0.0
    valid = [VALID_ACTIONS[i] for i, m in enumerate(mask)
             if m == 1 and i < len(VALID_ACTIONS)]
    turn_num = int(t[0] * env.config.max_turns)

    return (
        f"Turn {turn_num}/{env.config.max_turns} | Crisis: {c[0]:.2f}\n"
        f"My Resources: {s[0]:.3f} | My Stability: {s[1]:.3f} "
        f"| Exposure: {s[2]:.3f} | Delta: {s[3]:+.3f}\n"
        f"Avg Peer Resources: {avg_peer_res:.3f} | Avg Peer Stability: {avg_peer_stab:.3f}\n"
        f"Valid actions: {', '.join(valid)}\n\nYour action:"
    )


def parse_action(text: str) -> str:
    """
    Extract the first valid action from a model's generated text.
    Falls back to 'ignore' if no valid action is found.
    """
    text = text.strip().lower().split('\n')[0]
    for action in VALID_ACTIONS:
        if action in text:
            return action
    return "ignore"


def load_frozen_opponent(checkpoint_path: str, use_cpu_fallback: bool = False):
    """
    Load a checkpoint as a frozen model for opponent inference.
    Uses float16 on GPU, float32 on CPU fallback.

    Args:
        checkpoint_path: path to HuggingFace checkpoint directory or model name
        use_cpu_fallback: if True, load on CPU (slower but uses no GPU VRAM)

    Returns:
        (model, tokenizer) — caller must call free_frozen_model(model) when done
    """
    device = "cpu" if use_cpu_fallback else "cuda"
    dtype  = torch.float32 if use_cpu_fallback else torch.float16

    frozen_tok = AutoTokenizer.from_pretrained(
        checkpoint_path,
        trust_remote_code=True,
    )
    if frozen_tok.pad_token is None:
        frozen_tok.pad_token = frozen_tok.eos_token

    frozen_model = AutoModelForCausalLM.from_pretrained(
        checkpoint_path,
        torch_dtype=dtype,
        device_map=device,
        trust_remote_code=True,
    )
    frozen_model.eval()  # no dropout

    return frozen_model, frozen_tok


@torch.no_grad()
def frozen_inference(frozen_model, frozen_tok, prompt: str,
                     max_new_tokens: int = 8) -> str:
    """
    Run inference on a frozen model. No gradients computed.

    Args:
        frozen_model: loaded frozen opponent model
        frozen_tok: associated tokenizer
        prompt: full prompt string
        max_new_tokens: max tokens to generate

    Returns:
        Parsed action string (guaranteed to be in VALID_ACTIONS)
    """
    device = next(frozen_model.parameters()).device
    inputs = frozen_tok(prompt, return_tensors="pt", truncation=True,
                        max_length=1800).to(device)
    output = frozen_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.5,
        do_sample=True,
        pad_token_id=frozen_tok.pad_token_id,
    )
    generated = frozen_tok.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return parse_action(generated)


print('✅ Frozen opponent infrastructure ready')

In [ ]:
# ============================================================
# CELL 7: Self-Play Episode Runner
# ============================================================
# Runs ONE complete episode where:
#   - District TRAIN_AGENT_ID is played by the training model
#   - All other districts are played by a SINGLE frozen opponent
#     sampled from the league (shared across all opponent slots)
#
# Returns:
#   - trajectory: list of (prompt, action, reward) for District 0 only
#   - episode_rewards: Dict[int, float] — cumulative reward per agent
#   - survived: bool — did District 0 survive all turns?

def run_league_episode(
    train_model,
    league: GenerationLeague,
    env: DistrictAccordEnv,
    env_parser: ActionParser,
    train_agent_id: int = TRAIN_AGENT_ID,
    current_gen: int = 0,
) -> Tuple[List[Dict], Dict[int, float], bool, int]:
    """
    Run one self-play episode against a frozen league opponent.

    Args:
        train_model: the model being trained (with gradients)
        league: GenerationLeague pool to sample opponents from
        env: DistrictAccordEnv instance
        env_parser: ActionParser for the environment
        train_agent_id: which district the training LLM controls
        current_gen: current training generation (for diversity tracking)

    Returns:
        trajectory: list of {"prompt": str, "action": str, "reward": float}
        episode_rewards: {agent_id: cumulative_reward}
        survived: whether training agent survived all turns
        opponent_gen: which generation was sampled as opponent
    """
    # Sample ONE opponent from league (shared for all 11 opponent slots)
    opponent_entry = league.sample_opponent(strategy="weighted_recent")
    league.record_sample(current_gen, opponent_entry.gen_num)

    # Load frozen opponent — VRAM aware
    frozen_model, frozen_tok = None, None
    use_cpu = False
    try:
        vram_before = vram_gb()
        frozen_model, frozen_tok = load_frozen_opponent(
            opponent_entry.checkpoint_path, use_cpu_fallback=False
        )
    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f'  ⚠ VRAM OOM — falling back to CPU for frozen opponent')
            torch.cuda.empty_cache()
            try:
                frozen_model, frozen_tok = load_frozen_opponent(
                    opponent_entry.checkpoint_path, use_cpu_fallback=True
                )
                use_cpu = True
            except Exception as e2:
                print(f'  ✗ Failed to load frozen model on CPU: {e2}')
                # Final fallback: use SelfPlayPolicy (no model)
                fallback_policy = SelfPlayPolicy(mode="rule_based", seed=42)
                frozen_model = None
        else:
            raise

    # Determine the training model device
    train_device = next(train_model.parameters()).device

    # Reset env
    obs = env.reset()
    episode_rewards = {i: 0.0 for i in range(env.config.num_districts)}
    trajectory = []   # (prompt, action, reward) for train_agent_id only
    survived = True

    for turn in range(env.config.max_turns):
        if env._done:
            break

        actions: Dict[int, str] = {}

        # ── Training agent action (with gradients via sampling) ────────────
        try:
            train_prompt = SYSTEM_PROMPT + "\n\n" + obs_to_prompt(
                obs[train_agent_id], train_agent_id, env
            )
            # Use inference mode for speed (GRPO collects separately)
            with torch.no_grad():
                FastLanguageModel = None
                inputs = tokenizer(train_prompt, return_tensors="pt",
                                   truncation=True, max_length=1800).to(train_device)
                out = train_model.generate(
                    **inputs, max_new_tokens=8,
                    temperature=0.7, do_sample=True,
                    pad_token_id=tokenizer.pad_token_id,
                )
                gen_text = tokenizer.decode(
                    out[0][inputs["input_ids"].shape[1]:],
                    skip_special_tokens=True,
                )
            train_action = parse_action(gen_text)
            actions[train_agent_id] = train_action
        except Exception as e:
            print(f'  ✗ Train agent inference error turn {turn}: {e}')
            actions[train_agent_id] = "ignore"
            train_prompt = ""
            train_action = "ignore"

        # ── Frozen opponent actions for all other agents ───────────────────
        for agent_id in range(env.config.num_districts):
            if agent_id == train_agent_id:
                continue
            if env._collapsed.get(agent_id, False):
                continue
            try:
                opp_prompt = SYSTEM_PROMPT + "\n\n" + obs_to_prompt(
                    obs[agent_id], agent_id, env
                )
                if frozen_model is not None:
                    actions[agent_id] = frozen_inference(
                        frozen_model, frozen_tok, opp_prompt
                    )
                else:
                    # SelfPlayPolicy fallback
                    parsed = fallback_policy.act({agent_id: obs[agent_id]}, env)
                    actions[agent_id] = parsed[agent_id]["action_type"].name.lower()
            except Exception:
                actions[agent_id] = "ignore"

        # ── Step environment ───────────────────────────────────────────────
        try:
            parsed_actions = env_parser.parse_structured_safe(
                {k: v for k, v in actions.items()}
            )
            obs, rewards, done, trunc, info = env.step(parsed_actions)

            for agent_id, r in rewards.items():
                episode_rewards[agent_id] = episode_rewards.get(agent_id, 0.0) + float(r)

            # Record trajectory step for training agent
            trajectory.append({
                "prompt": train_prompt,
                "action": train_action,
                "reward": float(rewards.get(train_agent_id, 0.0)),
            })

            if done or trunc:
                break

        except Exception as e:
            print(f'  ✗ env.step error at turn {turn}: {e}')
            break

    # Check if training agent survived
    survived = not env._collapsed.get(train_agent_id, False)

    # Free frozen model VRAM
    if frozen_model is not None:
        free_frozen_model(
            frozen_model, f"Gen {opponent_entry.gen_num}"
        )

    return trajectory, episode_rewards, survived, opponent_entry.gen_num


print('✅ Self-play episode runner ready')

In [ ]:
# ============================================================
# CELL 8: GRPO Reward Function & Evaluation
# ============================================================

def league_reward_fn(prompts: list, completions: list, **kwargs) -> list:
    """
    Multi-component reward function for GRPO training.
    Same structure as the base training but also rewards:
    - Surviving longer (proportional to turns survived)
    - Cooperative behavior against strong opponents
    """
    rewards = []
    for prompt, completion in zip(prompts, completions):
        # Extract action text
        if hasattr(completion, 'text'):
            action = completion.text.strip().split("\n")[0].strip().lower()
        elif isinstance(completion, list):
            action = tokenizer.decode(completion, skip_special_tokens=True
                                      ).strip().split("\n")[0].strip().lower()
        else:
            action = str(completion).strip().split("\n")[0].strip().lower()

        score = 0.0
        base_action = action.split(":")[0]

        # Hard gate: invalid action
        if base_action not in VALID_ACTIONS:
            rewards.append(-1.0)
            continue

        score += 0.5   # valid action

        # Format compliance
        if len(action.split()) <= 2:
            score += 0.2
        else:
            score -= 0.3

        # Context-aware scoring
        try:
            crisis_val, resources, stability = 0.0, 0.5, 0.5
            for line in prompt.split("\n"):
                if "Crisis:" in line:
                    crisis_val = float(line.split("Crisis:")[-1].strip())
                if "My Resources:" in line:
                    for p in line.split("|"):
                        if "My Resources:" in p:
                            resources = float(p.split(":")[-1].strip())
                        elif "My Stability:" in p:
                            stability = float(p.split(":")[-1].strip())

            if crisis_val > 0.4 and base_action in ("defend", "recover"): score += 0.4
            if stability < 0.3 and base_action == "recover":               score += 0.3
            if resources > 0.4 and stability > 0.5 and base_action == "invest": score += 0.2
            if base_action in ("propose", "accept", "share"):              score += 0.3
            if base_action == "ignore":                                     score -= 0.2
        except (ValueError, IndexError):
            pass

        rewards.append(score)
    return rewards


def evaluate_generation(
    model,
    eval_seeds: List[int] = EVAL_SEEDS,
    gen_label: str = "?",
) -> Tuple[float, float, List[str]]:
    """
    Evaluate model on fixed seeds. No training, inference only.

    Returns:
        avg_reward: average cumulative reward for the training agent
        collapse_rate: fraction of episodes where training agent collapsed
        action_sequences: list of action strings per episode (for behavior comparison)
    """
    from unsloth import FastLanguageModel as FLM
    FLM.for_inference(model)  # fast inference mode

    device = next(model.parameters()).device
    all_rewards, all_collapsed, all_sequences = [], [], []

    for seed in eval_seeds:
        try:
            env = DistrictAccordEnv(ENV_CFG)
            parser = ActionParser(ENV_CFG)
            obs = env.reset(seed=seed)

            # Opponents: rule_based for fair evaluation
            opponent = SelfPlayPolicy(mode="rule_based", seed=seed)
            opp_actions_cache = {}

            cumulative_reward = 0.0
            action_seq = []

            for turn in range(ENV_CFG.max_turns):
                if env._done:
                    break

                # Training agent
                prompt = SYSTEM_PROMPT + "\n\n" + obs_to_prompt(
                    obs[TRAIN_AGENT_ID], TRAIN_AGENT_ID, env
                )
                with torch.no_grad():
                    inputs = tokenizer(prompt, return_tensors="pt",
                                       truncation=True, max_length=1800).to(device)
                    out = model.generate(
                        **inputs, max_new_tokens=8,
                        temperature=0.3, do_sample=True,
                        pad_token_id=tokenizer.pad_token_id,
                    )
                    gen_text = tokenizer.decode(
                        out[0][inputs["input_ids"].shape[1]:],
                        skip_special_tokens=True,
                    )
                action = parse_action(gen_text)
                action_seq.append(action)

                # Opponents
                opp_parsed = opponent.act(obs, env)
                all_actions = {**opp_parsed,
                               **parser.parse_structured_safe({TRAIN_AGENT_ID: action})}

                obs, rewards, done, trunc, info = env.step(all_actions)
                cumulative_reward += float(rewards.get(TRAIN_AGENT_ID, 0.0))

                if done or trunc:
                    break

            all_rewards.append(cumulative_reward)
            all_collapsed.append(env._collapsed.get(TRAIN_AGENT_ID, False))
            all_sequences.append(action_seq)

        except Exception as e:
            print(f'  ✗ Eval error (seed={seed}): {e}')
            all_rewards.append(0.0)
            all_collapsed.append(True)
            all_sequences.append([])

    # Switch back to training mode
    from unsloth import FastLanguageModel as FLM
    FLM.for_training(model)

    avg_reward    = float(np.mean(all_rewards))
    collapse_rate = float(np.mean(all_collapsed))

    print(f'  [Eval Gen {gen_label}] avg_reward={avg_reward:.3f}, "
          f"collapse_rate={collapse_rate:.1%}')

    return avg_reward, collapse_rate, all_sequences


print('✅ Reward function and evaluation ready')

In [ ]:
# ============================================================
# CELL 9: Generational Training Loop
# ============================================================
# Main self-play training loop:
#   For each generation:
#     1. Run EPISODES_PER_GEN league episodes → collect prompts
#     2. Train via GRPO on those prompts
#     3. Save checkpoint
#     4. Register in league
#     5. Evaluate on fixed seeds
#     6. Print generation summary

# Store results per generation
gen_results = []  # [{gen, avg_reward, collapse_rate, sequences}]

# Evaluate Gen 0 baseline first
print('=' * 60)
print('Evaluating Gen 0 baseline...')
print('=' * 60)
vram_report('gen 0 eval start')

gen0_reward, gen0_collapse, gen0_seqs = evaluate_generation(
    train_model, gen_label="0 (base)"
)
gen_results.append({
    "gen": 0, "avg_reward": gen0_reward,
    "collapse_rate": gen0_collapse, "sequences": gen0_seqs,
})

# Update Gen 0 league entry with eval results
league.generations[0].avg_eval_reward = gen0_reward
league.generations[0].collapse_rate   = gen0_collapse
league.save()

print(f'\nGen 0 baseline: avg_reward={gen0_reward:.3f}, collapse_rate={gen0_collapse:.1%}')
print()

# ── Self-play generations ─────────────────────────────────────────────────
for gen in range(1, NUM_GENERATIONS + 1):
    print('=' * 60)
    print(f'GENERATION {gen} / {NUM_GENERATIONS}')
    print('=' * 60)
    vram_report(f'gen {gen} start')

    gen_prompts = []   # will become GRPO training dataset
    gen_rewards = []   # episode-level reward for tracking

    # ── Phase 1: Collect trajectories via league episodes ─────────────────
    print(f'\n[Phase 1] Collecting {EPISODES_PER_GEN} league episodes...')
    survived_count = 0

    for ep in range(EPISODES_PER_GEN):
        try:
            env = DistrictAccordEnv(ENV_CFG)
            parser = ActionParser(ENV_CFG)

            traj, ep_rewards, survived, opp_gen = run_league_episode(
                train_model, league, env, parser,
                train_agent_id=TRAIN_AGENT_ID,
                current_gen=gen,
            )

            # Collect prompts from trajectory (all turns)
            for step in traj:
                gen_prompts.append({"prompt": step["prompt"]})

            train_ep_reward = ep_rewards.get(TRAIN_AGENT_ID, 0.0)
            gen_rewards.append(train_ep_reward)
            survived_count += int(survived)

            if ep % 5 == 0:
                print(f'  Ep {ep+1:2d}/{EPISODES_PER_GEN} | '
                      f'opp=Gen{opp_gen} | '
                      f'reward={train_ep_reward:.2f} | '
                      f'survived={survived}')

        except Exception as e:
            print(f'  ✗ Episode {ep} error: {e}')
            traceback.print_exc()

    collect_avg = float(np.mean(gen_rewards)) if gen_rewards else 0.0
    survival_rate = survived_count / EPISODES_PER_GEN
    print(f'\n  Collection done: avg_reward={collect_avg:.3f}, '
          f'survival={survival_rate:.1%}, prompts={len(gen_prompts)}')

    # ── Phase 2: GRPO training ─────────────────────────────────────────────
    print(f'\n[Phase 2] GRPO training on {len(gen_prompts)} prompts...')
    vram_report(f'gen {gen} before GRPO')

    if len(gen_prompts) < 4:
        print('  ⚠ Too few prompts — skipping GRPO update')
    else:
        dataset = Dataset.from_list(gen_prompts)
        ckpt_dir = os.path.join(SELFPLAY_BASE, f"gen_{gen}")

        training_args = GRPOConfig(
            output_dir=ckpt_dir,
            num_generations=4,
            max_completion_length=16,
            max_prompt_length=1800,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=2,
            num_train_epochs=1,
            learning_rate=3e-6,    # slightly lower lr for self-play stability
            logging_steps=5,
            save_steps=500,        # only save at end (reduces disk I/O)
            save_strategy="epoch",
            report_to="none",
            fp16=True,
            seed=gen * 100,        # different seed per generation
        )

        trainer = GRPOTrainer(
            model=train_model,
            tokenizer=tokenizer,
            reward_funcs=[league_reward_fn],
            args=training_args,
            train_dataset=dataset,
        )
        trainer.train()

        # Save checkpoint for league pool
        train_model.save_pretrained_merged(
            ckpt_dir, tokenizer, save_method="lora"
        )
        print(f'  ✅ Checkpoint saved: {ckpt_dir}')

    # ── Phase 3: Evaluate new generation ──────────────────────────────────
    print(f'\n[Phase 3] Evaluating Gen {gen} on {EVAL_EPISODES} fixed seeds...')
    ckpt_dir = os.path.join(SELFPLAY_BASE, f"gen_{gen}")

    gen_avg_reward, gen_collapse, gen_seqs = evaluate_generation(
        train_model, eval_seeds=EVAL_SEEDS, gen_label=str(gen)
    )

    gen_results.append({
        "gen": gen,
        "avg_reward": gen_avg_reward,
        "collapse_rate": gen_collapse,
        "sequences": gen_seqs,
    })

    # Register this generation in the league
    league.add_generation(
        gen_num=gen,
        checkpoint_path=ckpt_dir,
        avg_eval_reward=gen_avg_reward,
        collapse_rate=gen_collapse,
    )

    # Print generation summary
    delta = gen_avg_reward - gen_results[-2]["avg_reward"]
    print(f'\n  ━━━ Gen {gen} Summary ━━━')
    print(f'  Avg eval reward:   {gen_avg_reward:.4f}  (Δ {delta:+.4f})')
    print(f'  Collapse rate:     {gen_collapse:.1%}')
    print(f'  Episodes collected: {EPISODES_PER_GEN}')
    print(f'  League pool size:   {len(league.generations)}')
    vram_report(f'gen {gen} end')
    print()

print('=' * 60)
print('✅ All generations complete!')
print('=' * 60)

In [ ]:
# ============================================================
# CELL 10: Plots
# ============================================================
# Generates three plots and saves to outputs/:
#   1. selfplay_generation_delta.png — reward per generation
#   2. selfplay_league_diversity.png — opponent sampling distribution
#   3. selfplay_winrate_matrix.png   — head-to-head reward heatmap

os.makedirs("outputs", exist_ok=True)

gens       = [r["gen"] for r in gen_results]
avg_rewards = [r["avg_reward"] for r in gen_results]
collapse_rates = [r["collapse_rate"] for r in gen_results]

# ── Plot 1: Generation delta curve ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(gens, avg_rewards, 'o-', color="#60a5fa", lw=2.5,
        markersize=8, label="Avg Eval Reward")

for g, r, cr in zip(gens, avg_rewards, collapse_rates):
    ax.annotate(f"Gen {g}\n{r:.3f}",
                xy=(g, r), xytext=(0, 14),
                textcoords="offset points",
                ha="center", fontsize=9,
                color="#60a5fa")

# Shade the improvement area
ax.fill_between(gens, avg_rewards[0], avg_rewards,
                alpha=0.15, color="#6ee7b7")
ax.axhline(avg_rewards[0], color="#f87171", ls="--",
           lw=1.5, label=f"Gen 0 baseline ({avg_rewards[0]:.3f})")

ax.set_xlabel("Generation", fontsize=12)
ax.set_ylabel("Avg Eval Reward", fontsize=12)
ax.set_title("District Accord — League Self-Play: Reward per Generation",
             fontsize=13)
ax.set_xticks(gens)
ax.set_xticklabels([f"Gen {g}" for g in gens])
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/selfplay_generation_delta.png", dpi=150)
plt.show()
print('✅ Saved outputs/selfplay_generation_delta.png')

# ── Plot 2: League diversity (opponent sampling) ───────────────────────────
# How often was each past generation chosen as opponent?
all_gen_nums = sorted(set(g.gen_num for g in league.get_all_generations()))
training_gens = sorted(league.opponent_samples.keys())

if training_gens:
    fig, ax = plt.subplots(figsize=(10, 5))
    bar_width = 0.6 / max(len(all_gen_nums), 1)
    colors_div = plt.cm.Blues(np.linspace(0.3, 0.9, len(all_gen_nums)))

    bottom = np.zeros(len(training_gens))
    for i, opp_gen in enumerate(all_gen_nums):
        counts = [
            league.opponent_samples[tg].count(opp_gen)
            for tg in training_gens
        ]
        ax.bar([str(g) for g in training_gens], counts,
               bottom=bottom, label=f"vs Gen {opp_gen}",
               color=colors_div[i], edgecolor='white', lw=0.5)
        bottom += np.array(counts, dtype=float)

    ax.set_xlabel("Training Generation", fontsize=12)
    ax.set_ylabel("Episodes Played Against", fontsize=12)
    ax.set_title("League Diversity — Opponent Sampling per Generation",
                 fontsize=13)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("outputs/selfplay_league_diversity.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print('✅ Saved outputs/selfplay_league_diversity.png')

# ── Plot 3: Win-rate matrix ────────────────────────────────────────────────
# Approximate: use avg_eval_reward per generation as proxy for strength.
# Head-to-head: Gen N vs Gen M → reward advantage of Gen N.
n = len(gen_results)
matrix = np.zeros((n, n))
for i, ri in enumerate(gen_results):
    for j, rj in enumerate(gen_results):
        matrix[i, j] = ri["avg_reward"] - rj["avg_reward"]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    matrix,
    annot=True, fmt=".2f", cmap="RdYlGn",
    center=0,
    xticklabels=[f"Gen {r['gen']}" for r in gen_results],
    yticklabels=[f"Gen {r['gen']}" for r in gen_results],
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Head-to-Head Reward Advantage (row − column)", fontsize=12)
ax.set_xlabel("Opponent Generation")
ax.set_ylabel("Training Generation")
plt.tight_layout()
plt.savefig("outputs/selfplay_winrate_matrix.png", dpi=150)
plt.show()
print('✅ Saved outputs/selfplay_winrate_matrix.png')

In [ ]:
# ============================================================
# CELL 11: Before/After Behavioral Comparison
# ============================================================
# Compare action sequences from Gen 0 (base) vs final Gen.
# Shows qualitatively what the model learned:
#   - Did coalition proposals increase?
#   - Did IGNORE decrease?
#   - Did survival improve?

from collections import Counter

COMPARE_EPISODES = 3  # episodes to sample for comparison


def format_sequence_summary(sequences: List[List[str]], label: str) -> str:
    """Format action sequence statistics for display."""
    lines = [f"{'='*60}", f"{label}", f"{'='*60}"]

    all_actions = [a for seq in sequences for a in seq]
    counter = Counter(all_actions)
    total = max(len(all_actions), 1)

    lines.append(f"\nAction Distribution ({len(sequences)} episodes, {total} total actions):")
    for action, count in sorted(counter.items(), key=lambda x: -x[1]):
        pct = count / total * 100
        bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
        lines.append(f"  {action:20} {bar} {pct:5.1f}%")

    lines.append(f"\nCooperative actions (propose+accept+share): "
                 f"{sum(counter.get(a,0) for a in ('propose','accept','share'))/total*100:.1f}%")
    lines.append(f"Ignore rate: {counter.get('ignore',0)/total*100:.1f}%")

    lines.append(f"\nEpisode summaries:")
    for i, seq in enumerate(sequences[:COMPARE_EPISODES]):
        lines.append(f"  Ep {i+1}: {' → '.join(seq[:10])}{'...' if len(seq)>10 else ''}")

    return "\n".join(lines)


# Gen 0 (base) sequences — from eval
gen0_seqs = gen_results[0]["sequences"][:COMPARE_EPISODES]

# Final Gen sequences — from eval
final_gen_result = gen_results[-1]
final_seqs = final_gen_result["sequences"][:COMPARE_EPISODES]

# Build comparison text
comparison_text = [
    "District Accord — League Self-Play: Before/After Behavioral Comparison",
    f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    format_sequence_summary(
        gen0_seqs, f"GEN 0 (Base) — Avg reward: {gen_results[0]['avg_reward']:.3f}"
    ),
    "",
    format_sequence_summary(
        final_seqs,
        f"GEN {final_gen_result['gen']} (Final) — Avg reward: {final_gen_result['avg_reward']:.3f}"
    ),
    "",
    "IMPROVEMENT SUMMARY",
    "=" * 60,
    f"Reward delta:    {final_gen_result['avg_reward'] - gen_results[0]['avg_reward']:+.4f}",
    f"Collapse delta:  {final_gen_result['collapse_rate'] - gen_results[0]['collapse_rate']:+.1%}",
    f"Generations run: {NUM_GENERATIONS}",
]

comparison_str = "\n".join(comparison_text)
print(comparison_str)

# Save to file
with open("outputs/selfplay_behavior_comparison.txt", "w") as f:
    f.write(comparison_str)
print("\n✅ Saved outputs/selfplay_behavior_comparison.txt")

In [ ]:
# ============================================================
# CELL 12: Final Summary Table
# ============================================================
# Prints the full generation summary and verifies all outputs exist.

def diversity_score(gen_num: int) -> float:
    """
    Diversity score = entropy over opponent generation distribution.
    1.0 = perfectly uniform, 0.0 = always same opponent.
    """
    samples = league.opponent_samples.get(gen_num, [])
    if not samples:
        return 0.0
    counts = np.array(list(Counter(samples).values()), dtype=float)
    probs = counts / counts.sum()
    entropy = -np.sum(probs * np.log2(probs + 1e-9))
    max_entropy = np.log2(len(league.generations))
    return float(entropy / max_entropy) if max_entropy > 0 else 0.0


print("\n" + "=" * 75)
print("  DISTRICT ACCORD — LEAGUE SELF-PLAY FINAL SUMMARY")
print("=" * 75)
print(f"{'Gen':>5} {'Avg Reward':>12} {'vs Gen0 Δ':>10} "
      f"{'Collapse%':>10} {'Diversity':>10}")
print("-" * 75)

baseline_reward = gen_results[0]["avg_reward"]
for r in gen_results:
    delta = r["avg_reward"] - baseline_reward
    div   = diversity_score(r["gen"])
    print(f"  {r['gen']:>3}   {r['avg_reward']:>12.4f}  "
          f"{delta:>+10.4f}  "
          f"{r['collapse_rate']:>9.1%}  "
          f"{div:>9.2f}")

print("-" * 75)
print(f"  Best gen: {max(gen_results, key=lambda x: x['avg_reward'])['gen']}")
print(f"  Total improvement: "
      f"{gen_results[-1]['avg_reward'] - baseline_reward:+.4f}")
print("=" * 75)

# Verify output files
expected_outputs = [
    "outputs/selfplay_generation_delta.png",
    "outputs/selfplay_league_diversity.png",
    "outputs/selfplay_winrate_matrix.png",
    "outputs/selfplay_behavior_comparison.txt",
    "outputs/league_state.json",
]
for f in expected_outputs:
    exists = "✅" if os.path.exists(f) else "❌"
    size = f"{os.path.getsize(f):,} bytes" if os.path.exists(f) else "missing"
    print(f"  {exists} {f:55} {size}")

print()
print("Self-play checkpoints:")
for gen in range(1, NUM_GENERATIONS + 1):
    ckpt = f"outputs/selfplay_checkpoints/gen_{gen}"
    exists = "✅" if os.path.exists(ckpt) else "❌"
    print(f"  {exists} {ckpt}")

In [ ]:
# ============================================================
# CELL 13: Save & Download
# ============================================================
# Zip everything and download from Colab.
# The training model's final LoRA adapter is also saved.

# Save final trained model
final_path = "outputs/selfplay_final_model"
train_model.save_pretrained_merged(
    final_path, tokenizer, save_method="lora"
)
print(f'✅ Final model saved to {final_path}/')

# Zip outputs (exclude large checkpoints to keep download small)
!zip -r selfplay_outputs.zip outputs/ \
    --exclude 'outputs/selfplay_checkpoints/*/adapter_model.safetensors' \
    --exclude 'outputs/grpo_checkpoints/*' \
    --quiet

print('\nOutput files included in download:')
!unzip -l selfplay_outputs.zip | grep -v '__MACOSX' | tail -30

from google.colab import files
files.download('selfplay_outputs.zip')
print('\n✅ Download started!')